# Fusion Weight Optimization


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
import itertools

train_probs = pd.read_csv("/content/fusion_train_probs.csv")
holdout_probs = pd.read_csv("/content/fusion_holdout_probs.csv")
print(f"Train: {len(train_probs)} | Holdout: {len(holdout_probs)}")

Train: 1196 | Holdout: 299


In [ ]:
def fuse(w, p_rgb, p_thermal, p_plantar):
    return w[0] * p_rgb + w[1] * p_thermal + w[2] * p_plantar


def score_weights(w, df, metric="auc"):
    p = fuse(w, df["p_rgb"].values, df["p_thermal_risk"].values, df["p_plantar_risk"].values)
    y = df["fused_label"].values
    if metric == "auc":
        return roc_auc_score(y, p)
    elif metric == "f1":
        return f1_score(y, (p >= 0.5).astype(int))
    elif metric == "acc":
        return accuracy_score(y, (p >= 0.5).astype(int))

## 1. Coarse grid search (step = 0.05, simplex constraint w1+w2+w3=1)

In [ ]:
step = 0.05
grid_vals = np.arange(0.0, 1.0 + step, step)

best_grid_score = -1
best_grid_w = None

for w1 in grid_vals:
    for w2 in grid_vals:
        w3 = round(1.0 - w1 - w2, 10)
        if w3 < 0 or w3 > 1:
            continue
        w = (w1, w2, w3)
        s = score_weights(w, train_probs, metric="auc")
        if s > best_grid_score:
            best_grid_score = s
            best_grid_w = w

print(f"Grid search best weights (w_rgb, w_thermal, w_plantar): {best_grid_w}")
print(f"Grid search best train AUC: {best_grid_score:.4f}")

Grid search best weights (w_rgb, w_thermal, w_plantar): (np.float64(0.9), np.float64(0.05), np.float64(0.05))
Grid search best train AUC: 0.9897


## 2. Gradient-based refinement (softmax-parameterized, keeps simplex constraint)

In [ ]:
def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()


def neg_auc_from_z(z, df):
    w = softmax(z)
    return -score_weights(w, df, metric="auc")


z0 = np.log(np.array(best_grid_w) + 1e-8)
res = minimize(neg_auc_from_z, z0, args=(train_probs,), method="Nelder-Mead",
                options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 2000})

refined_w = softmax(res.x)
refined_score = score_weights(refined_w, train_probs, metric="auc")

print(f"Refined weights (w_rgb, w_thermal, w_plantar): {refined_w}")
print(f"Refined train AUC: {refined_score:.4f}")

final_w = refined_w if refined_score > best_grid_score else np.array(best_grid_w)
print(f"\nFinal chosen weights: {final_w}")

Refined weights (w_rgb, w_thermal, w_plantar): [0.89829817 0.0552593  0.04644253]
Refined train AUC: 0.9898

Final chosen weights: [0.89829817 0.0552593  0.04644253]


## 3. Stratified 5-Fold CV — weight stability check

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_idx = train_probs.index.values
y = train_probs["fused_label"].values

fold_weights = []
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_idx, y), start=1):
    tr_df = train_probs.iloc[tr_idx]
    val_df = train_probs.iloc[val_idx]

    best_s, best_w = -1, None
    for w1 in grid_vals:
        for w2 in grid_vals:
            w3 = round(1.0 - w1 - w2, 10)
            if w3 < 0 or w3 > 1:
                continue
            s = score_weights((w1, w2, w3), tr_df, metric="auc")
            if s > best_s:
                best_s, best_w = s, (w1, w2, w3)

    val_score = score_weights(best_w, val_df, metric="auc")
    fold_weights.append(best_w)
    fold_scores.append(val_score)
    print(f"Fold {fold}: weights={best_w} | val AUC={val_score:.4f}")

fold_weights = np.array(fold_weights)
print(f"\nMean weights: {fold_weights.mean(axis=0)} +/- {fold_weights.std(axis=0)}")
print(f"Mean val AUC: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")

Fold 1: weights=(np.float64(0.9), np.float64(0.05), np.float64(0.05)) | val AUC=0.9928
Fold 2: weights=(np.float64(0.9), np.float64(0.05), np.float64(0.05)) | val AUC=0.9974
Fold 3: weights=(np.float64(0.9), np.float64(0.05), np.float64(0.05)) | val AUC=0.9957
Fold 4: weights=(np.float64(0.9), np.float64(0.05), np.float64(0.05)) | val AUC=0.9820
Fold 5: weights=(np.float64(0.9), np.float64(0.05), np.float64(0.05)) | val AUC=0.9775

Mean weights: [0.9  0.05 0.05] +/- [0. 0. 0.]
Mean val AUC: 0.9891 +/- 0.0079


## 4. Final held-out evaluation — fused vs. single-modality baselines

In [ ]:
def evaluate(y_true, y_prob, name):
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}% | AUC: {auc:.4f} | F1: {f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=["No DFU", "DFU"]))
    return {"name": name, "accuracy": acc, "auc": auc, "f1": f1}


y_true = holdout_probs["fused_label"].values
p_fused_final = fuse(final_w, holdout_probs["p_rgb"].values,
                      holdout_probs["p_thermal_risk"].values,
                      holdout_probs["p_plantar_risk"].values)

results = []
results.append(evaluate(y_true, holdout_probs["p_rgb"].values, "RGB only"))
results.append(evaluate(y_true, holdout_probs["p_thermal_risk"].values, "Thermal only"))
results.append(evaluate(y_true, holdout_probs["p_plantar_risk"].values, "Plantar only"))
results.append(evaluate(y_true, p_fused_final, f"Fused (w={np.round(final_w,3)})"))

results_df = pd.DataFrame(results)
results_df.to_csv("/content/final_fusion_results.csv", index=False)
print("\nSaved /content/final_fusion_results.csv")
results_df


--- RGB only ---
Accuracy: 94.65% | AUC: 0.9697 | F1: 0.9681
              precision    recall  f1-score   support

      No DFU       0.95      0.74      0.83        54
         DFU       0.95      0.99      0.97       245

    accuracy                           0.95       299
   macro avg       0.95      0.87      0.90       299
weighted avg       0.95      0.95      0.94       299


--- Thermal only ---
Accuracy: 78.93% | AUC: 0.7317 | F1: 0.8645
              precision    recall  f1-score   support

      No DFU       0.44      0.65      0.53        54
         DFU       0.91      0.82      0.86       245

    accuracy                           0.79       299
   macro avg       0.68      0.73      0.70       299
weighted avg       0.83      0.79      0.80       299


--- Plantar only ---
Accuracy: 76.25% | AUC: 0.6820 | F1: 0.8453
              precision    recall  f1-score   support

      No DFU       0.40      0.63      0.49        54
         DFU       0.91      0.79      0.85

,name,accuracy,auc,f1
0,RGB only,0.946488,0.969690,0.968127
1,Thermal only,0.789298,0.731708,0.864516
2,Plantar only,0.762542,0.682011,0.845316
3,Fused (w=[0.898 0.055 0.046]),0.943144,0.985639,0.966203
